# 생성형 AI 윤리 가이드북 QA (Reranker)

01 프로젝트에서 만든 **FAISS + Cross Encoder Reranker + 시스템 프롬프트** RAG 파이프라인을 새로운 문서인 `생성형_AI윤리_가이드북_RAG시험용_30쪽.pdf`에 적용했다. 질문은 코드에 직접 쓰지 않고, 학생별로 나눠 받은 질문 파일(`students/student_11_questions.json`)에서 불러온다.

| 항목 | 01 프로젝트 (사내 업무 가이드) | 02 프로젝트 (AI 윤리 가이드북) |
|---|---|---|
| 문서 | 사내 규정 PDF 9쪽 | 법·윤리 서술형 PDF 30쪽 |
| 질문 | 코드에 직접 작성한 10개 | JSON 파일에서 불러온 3개 (난이도 하·중·상) |
| 1차 검색 후보 수 | `k=7` | `k=8` |
| Reranker 최종 문서 수 | `top_n=3` | `top_n=4` |
| 패키지 관리 | pip | uv |

## 1. 실험 설정

청크 크기와 겹침, 사용할 질문 파일의 학생 번호를 맨 위에 모아둬서 값만 바꿔 다시 실행할 수 있게 했다.

In [36]:
# PDF를 자를 청크 사이즈와 오버랩 사이즈 결정
CHUNK_SIZE=500
CHUNK_OVERLAP=100

# 학생 번호(내 번호는 11번)
STUDENT_NUMBER=11

## 2. 질문 불러오기

질문 파일은 `question`(질문), `difficulty`(난이도), `reference`(참고 답안), `source_pages`(정답 근거 페이지) 등을 담고 있다. RAG에는 질문만 넣으므로 `question`만 뽑아서 리스트로 만든다.

In [37]:
# 학생 번호 11번 질문 데이터 불러오기 
import json
with open(f"students/student_{STUDENT_NUMBER}_questions.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [38]:
# JSON에서 질문 리스트만 추출
query_list = [question.get("question") for question in data["questions"]]

## 3. 환경변수 로드

In [39]:
# 환경변수와 프로젝트 이름 설정
from dotenv import load_dotenv 
import os

load_dotenv()
os.environ["LANGSMITH_PROJECT"] = "RAG Test"

## 4. 검색 결과 출력 함수

검색된 문서를 번호와 페이지(`page_label`)를 붙여 구분선으로 나눠 출력한다. 검색이 정답 페이지를 제대로 찾았는지 눈으로 확인하는 용도다.

In [40]:
# 리트리버 결과 이쁘게 읽는 함수
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1} (p.{d.metadata.get("page_label")}):\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

## 5. 문서 로드 → 분할 → 임베딩 → FAISS 저장

`PyPDFLoader`로 30쪽 PDF를 페이지 단위로 읽고, `RecursiveCharacterTextSplitter`(500자, 겹침 100자)로 나눈 뒤 `text-embedding-3-small`로 임베딩해서 FAISS에 저장한다.

In [41]:
from langchain_community.document_loaders import PyPDFLoader 
from langchain_community.vectorstores import FAISS 
from langchain_openai import OpenAIEmbeddings 
from langchain_text_splitters import RecursiveCharacterTextSplitter

# PDF 로드 
loader = PyPDFLoader("data/생성형_AI윤리_가이드북_RAG시험용_30쪽.pdf")
documents = loader.load() 

# 문서 분할 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

split_docs = text_splitter.split_documents(documents) 

# 임베딩 
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# FAISS 벡터스토어 생성 
db = FAISS.from_documents(split_docs, embeddings)

## 6. 리트리버와 Reranker 구성

문서가 9쪽에서 30쪽으로 길어져서, 1차 검색 후보를 `k=8`로 늘리고 Reranker가 최종으로 고르는 문서도 `top_n=4`로 늘렸다. 정답 근거가 여러 페이지에 나뉘어 있는 질문도 있어서, 후보 풀과 최종 문서 수를 조금씩 넉넉하게 잡았다.

- 1차 검색: FAISS 유사도 검색으로 후보 8개
- 2차 재정렬: Cross Encoder(`BAAI/bge-reranker-v2-m3`)가 질문-문서 쌍의 관련도를 직접 계산해서 상위 4개 선택

In [42]:
# 리트리버 생성
retriever = db.as_retriever(search_kwargs={"k":8})

In [ ]:
# 허깅페이스 모델(리랭커) 한 번만 불러오기
from langchain_community.cross_encoders import HuggingFaceCrossEncoder 
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [44]:
from langchain_classic.retrievers import ContextualCompressionRetriever 
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker 

# 리랭커 생성
compressor = CrossEncoderReranker(model=model, top_n=4)

# 리트리버, 리랭커 합체
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever,
)

# 각 질문에 대하여 리트리버 생성 결과 출력
for i, query in enumerate(query_list):
    compressed_docs = compression_retriever.invoke(query)
    print(f"\n질문{i+1}. {query}\n")
    pretty_print_docs(compressed_docs)


질문1. 생성형 AI를 이용해 가짜 뉴스를 만들고 배포했을 때, 피해를 줄 목적이 없더라도 법적 책임이 발생할 수 있는 경우는 어떤 상황인가요?

Document 1 (p.16):

실존하는 개인 혹은 법인의 명예를 훼손하는 내용을 다루고 있거나, 이러한 내용으로 인해 상대방의 
명예나 인격이 손상될 수 있다는 점을 인식하고 있다면(혹은 더 나아가 명예훼손 등의 고의가 있다면), 
이는 관련 법령에 따른 형사 처벌 대상이 될 수 있음은 물론, 민사상 손해배상 청구 소송의 대상이 될 수도 
있어요.
다시 말해 생성형 AI 기술을 이용해 사진이나 영상을 제작하고, 이를 토대로 가짜 뉴스 영상을 제작하는 
행위는 그 자체로 하나의 창작적인 행위라고 볼 수 있지만, 이렇게 제작된 영상을 마치 실제 내용을 담고 
있는 뉴스 영상인 것처럼 배포하는 경우에는 법적인 책임을 부담할 수도 있습니다.
최근 미국에서 생성형 AI 기술을 활용해 미국 국방부(펜타곤) 청사 근처에서 대형 폭발이 일어났다고 
주장하는 사진이 SNS를 통해 확산되었는데, 이로 인해 미국 주가가 일시적으로 하락하는 등의 사건이 
발생해 문제가 된 적이 있어요.
----------------------------------------------------------------------------------------------------
Document 2 (p.16):

43
  
   
             
               
             
               
               
       
                 
             
  
   
              
            
              
      
                
             
             
               
      
1    재미로 생성형 AI를 이용해서 가짜 뉴스를 만들어 배포했어요. 누군가에게 
피해를 

## 7. 답변 생성

01 프로젝트와 같은 시스템 프롬프트로, 검색된 문서에 있는 내용만으로 답하고 근거 페이지를 밝히게 한다. 문서에 없는 내용은 "문서에 없음"이라고 답하도록 지시한다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate 
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 리트리버 결과를 포맷하는 함수(프롬프트에 넣을 검색 결과)
def format_docs(docs):
    context = f"\n{'-' * 100}\n".join(
        [f"(p.{d.metadata.get('page_label')}):\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    return context

# 프롬프트 템플릿 생성    
prompt = ChatPromptTemplate.from_messages([
    # 시스템 프롬프트(규칙)
    # 리트리버 결과(컨텍스트)
    # 사용자 질문(실제 질문)
    ("system", """
    1. 당신은 사용자의 질문에 대답을 하는 친절한 AI agent이다.
    2. 반드시 문서에 있는 내용만으로 답한다.
    3. 사용자의 질문을 문서에서 찾을 수 없으면 문서에서 찾지 못하여 대답을 할 수 없다고 답변한다.
    4. 질문이 여러 항목이면 항목마다 문서에 있는지 판단하고, 없는 항목은 '문서에 없음'이라고 밝힌다.
    5. 답변에 근거 페이지를 밝힌다.

    문서:
    {context}"""),
    ("human", "{question}"),
])

# LLM 생성(최신 경량 모델인 GPT 5.6 Luna 사용)
llm = ChatOpenAI(model="gpt-5.6-luna", temperature=0)

# 체인 생성
rag_chain = prompt | llm | StrOutputParser()

# 각 질문에 대하여 답변 생성
for i, query in enumerate(query_list):
    compressed_docs = compression_retriever.invoke(query)
    context = format_docs(compressed_docs)
    answer = rag_chain.invoke({"context": context, "question": query})
    print(f"질문 {i+1}: {query}\n\n답변: {answer}\n\n{'-'*80}\n\n")

질문 1: 생성형 AI를 이용해 가짜 뉴스를 만들고 배포했을 때, 피해를 줄 목적이 없더라도 법적 책임이 발생할 수 있는 경우는 어떤 상황인가요?

답변: 피해를 줄 목적이 없더라도 다음과 같은 경우 법적 책임이 발생할 수 있습니다.

- **실존하는 개인이나 법인의 명예를 훼손하는 내용을 다룬 경우**: 형사 처벌이나 민사상 손해배상 청구의 대상이 될 수 있습니다. (p.16)
- **가짜 뉴스를 실제 뉴스처럼 배포하여 누군가에게 직접적인 손해가 발생한 경우**: 민사상 손해배상 책임을 질 수 있고, 경우에 따라 특별법에 따른 처벌을 받을 수도 있습니다. (p.17)
- **제작 과정에서 실존 인물의 사진이나 영상을 허락 없이 사용한 경우**: 초상권 침해에 따른 민사상 손해배상 책임이 발생할 수 있습니다. (p.17)

다만, 세상에 존재하지 않는 내용을 바탕으로 가짜 뉴스를 만들었다는 사실만으로는 그 자체가 위법하다고 볼 수 없다고 설명되어 있습니다. (p.16)

--------------------------------------------------------------------------------


질문 2: 콜롬비아 판사가 판결문 작성에 챗GPT를 활용한 것에 대해 후안 다비드 구티에레스 교수는 어떤 입장을 보였나요?

답변: 후안 다비드 구티에레스 교수는 판사가 챗GPT를 활용한 것을 **확실히 비윤리적이며 무책임한 행동**이라고 비판했습니다. 또한 챗GPT에 동일한 질문을 해도 서로 다른 답변을 받았다고 지적했습니다.  
**근거: 문서 p.14**

--------------------------------------------------------------------------------


질문 3: 생성형 AI에 개인정보나 회사 기밀을 입력할 경우 발생할 수 있는 보안 문제와 그 이유를 설명하세요.

답변: 생성형 AI에 개인정보나 회사 기밀을 입력하면 다음과 같은 보안 문제가 발생할 수 있습니다.

- **기업 서버에 저장될 

## 8. 결과 확인

질문 파일에 들어 있는 참고 답안(`reference`)과 정답 근거 페이지(`source_pages`)를 기준으로 채점했다.

| # | 난이도 | 질문 요지 | 정답 근거 페이지 | Reranker 상위 4개 페이지 | 답변 | 판정 |
|---|---|---|---|---|---|---|
| 1 | 중 | 가짜 뉴스 배포 시 법적 책임 | p.16 | 16, 16, 17, 10 | 명예훼손 시 형사 처벌·민사 손해배상 (p.16) + 초상권 침해 등 추가 사례 (p.17) | 정확 |
| 2 | 하 | 콜롬비아 판사의 챗GPT 활용에 대한 교수의 입장 | p.14 | 14, 26, 27, 26 | "확실히 비윤리적이며 무책임한 행동", 같은 질문에 다른 답변을 받았다고 지적 (p.14) | 정확 |
| 3 | 상 | 개인정보·기밀 입력 시 보안 문제 | p.16, p.23 | 23, 20, 23, 14 | 서버 저장, 학습 재이용, 외부 유출 위험 (p.20, p.23) | 정확 |

- 세 질문 모두 Reranker가 고른 **1위 문서가 정답 근거 페이지**였고, 답변 내용도 참고 답안과 일치했다(3/3).
- 3번은 정답 근거가 p.16과 p.23 두 곳인데 p.16은 상위 4개에 들지 못했다. 그래도 p.23과 p.20에 같은 내용(서버 저장·학습 재이용·유출 위험)이 있어서 답변에는 문제가 없었다. 근거가 여러 페이지에 흩어진 질문에서는 `top_n`을 늘리거나 청크 크기를 조절해볼 여지가 있다.
- 1번 답변은 참고 답안보다 넓게 초상권 침해(p.17)까지 덧붙였는데, 이것도 문서에 있는 내용이라 환각(hallucination)은 아니다.